In [1]:
import pandas as pd
import numpy as np
from lightgbm import LGBMClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score


In [2]:
df = pd.read_csv('/home/tori/behavioural-fruad-detection/data/raw/train.csv')
df2 = pd.read_csv('/home/tori/behavioural-fruad-detection/data/raw/test.csv')

df['is_test'] = 0
df2['is_test'] = 1

full = pd.concat([df, df2])
full = full.sort_values(by=["user_id", "timestamp"]).reset_index(drop=True)
full.shape

(227656, 11)

In [3]:
full['is_first_txn'] = (full['hours_since_prev_txn'] == 999).astype(int)
full['hours_since_prev_txn'] = full['hours_since_prev_txn'].replace(999, np.nan)


In [4]:
full['running_total'] = full.groupby('user_id')['amount'].cumsum()

full['past_total'] = full['running_total'] - full['amount'] 
full['past_count'] = full.groupby('user_id').cumcount()
full['past_avg_amount'] = full['past_total'] / full['past_count'] 
full['amount_ratio'] = full['amount'] / full['past_avg_amount']

filtered_user = full[full['user_id'] == 1818]
filtered_user[['amount', 'past_avg_amount', 'amount_ratio']]

full = full.drop(columns=['running_total', 'past_total', 'past_count'])

In [5]:
filtered_user[['amount', 'past_avg_amount', 'amount']]

,amount,past_avg_amount,amount
82574,17.788061,NaN,17.788061
82575,33.010191,17.788061,33.010191
82576,37.474351,25.399126,37.474351
82577,15.018893,29.424201,15.018893
82578,10.954236,25.822874,10.954236
82579,5.410091,22.849147,5.410091
82580,14.351851,19.942637,14.351851
82581,10.819577,19.143953,10.819577
82582,5.095317,18.103406,5.095317
82583,16.608058,16.658063,16.608058


In [6]:
full['country_seen_count'] = full.groupby(['user_id', 'country']).cumcount()
full['is_new_country'] = (full['country_seen_count'] == 0).astype(int)

full['merchant_seen_count'] = full.groupby(['user_id', 'merchant_category']).cumcount()
full['is_new_merchant'] = (full['merchant_seen_count'] == 0).astype(int)

In [7]:
full['is_new_country'].sum()

np.int64(22571)

In [8]:
full['is_new_merchant'].sum()

np.int64(57058)

In [9]:
filtered_user = full[full['user_id'] == 1818]
filtered_user[['country', 'is_new_country', 'merchant_category', 'is_new_merchant']]

,country,is_new_country,merchant_category,is_new_merchant
82574,4,1,11,1
82575,4,0,11,0
82576,4,0,0,1
82577,4,0,11,0
82578,4,0,11,0
82579,12,1,0,0
82580,12,0,0,0
82581,12,0,11,0
82582,12,0,11,0
82583,12,0,11,0


In [10]:
full = full.drop(columns=['merchant_seen_count', 'country_seen_count'])

In [11]:
full['prev_gap_1'] = full.groupby('user_id')['hours_since_prev_txn'].shift(1)
full['prev_gap_2'] = full.groupby('user_id')['hours_since_prev_txn'].shift(2)

full['avg_gap_last3'] = full[['hours_since_prev_txn', 'prev_gap_1', 'prev_gap_2']].mean(axis=1)

In [12]:
filtered_user = full[full['user_id'] == 1818]
filtered_user[['hours_since_prev_txn', 'prev_gap_1', 'prev_gap_2', 'avg_gap_last3']]

,hours_since_prev_txn,prev_gap_1,prev_gap_2,avg_gap_last3
82574,NaN,NaN,NaN,NaN
82575,97765.0,NaN,NaN,97765.000000
82576,16471.0,97765.0,NaN,57118.000000
82577,66443.0,16471.0,97765.0,60226.333333
82578,90995.0,66443.0,16471.0,57969.666667
82579,76486.0,90995.0,66443.0,77974.666667
82580,2086.0,76486.0,90995.0,56522.333333
82581,55617.0,2086.0,76486.0,44729.666667
82582,35832.0,55617.0,2086.0,31178.333333
82583,26606.0,35832.0,55617.0,39351.666667


In [13]:
full = full.drop(columns=['prev_gap_1', 'prev_gap_2'])

In [14]:
labeled = full[full['is_test'] == 0]
unlabeled = full[full['is_test'] == 1]

In [15]:
labeled.shape
unlabeled.shape

(45531, 17)

In [16]:
full.columns

Index(['transaction_id', 'user_id', 'timestamp', 'amount', 'merchant_category',
       'country', 'device_id', 'channel', 'hours_since_prev_txn', 'label',
       'is_test', 'is_first_txn', 'past_avg_amount', 'amount_ratio',
       'is_new_country', 'is_new_merchant', 'avg_gap_last3'],
      dtype='str')

In [17]:
full = full.drop(columns=['prev_gap_2'])

KeyError: "['prev_gap_2'] not found in axis"

In [ ]:
full.columns

In [ ]:
labeled = full[full['is_test'] == 0]
unlabeled = full[full['is_test'] == 1]

In [ ]:
labeled.shape
unlabeled.shape

In [ ]:
features = ['amount', 'channel', 'hours_since_prev_txn', 'is_first_txn', 'past_avg_amount', 'amount_ratio', 'is_new_device', 'is_new_country', 'is_new_merchant', 'avg_gap_last3']
x = labeled[features]
y = labeled['label']

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=42, stratify=y
)

x_train.shape
x_test.shape
y_train.mean()
y_test.mean()


In [ ]:
model = LGBMClassifier(
    random_state=42,
    verbose=-1  
)

model.fit(x_train, y_train)

y_prob = model.predict_proba(x_test)[:, 1]

In [ ]:
y_prob[:10]

In [ ]:
average_precision_score(y_test, y_prob)

In [ ]:
labeled.groupby('label')[features].mean()

In [ ]:
y_prob_train = model.predict_proba(x_train)[:, 1]
average_precision_score(y_train, y_prob_train)

In [ ]:
cutoff = labeled['amount_ratio'].quantile(0.99)
spikes = labeled[labeled['amount_ratio'] >= cutoff]
spikes['label'].mean()


In [ ]:
y_prob_train = model.predict_proba(X_train)[:, 1]
average_precision_score(y_train, y_prob_train)

In [ ]:
model2 = LGBMClassifier(
    random_state=42,
    verbose=-1,
    num_leaves=15,
    min_child_samples=200,
    learning_rate=0.05,
    n_estimators=200
)
model2.fit(X_train, y_train)
y_prob2 = model2.predict_proba(X_test)[:, 1]
average_precision_score(y_test, y_prob2)

In [ ]:
model2.feature_importances_

In [ ]:
y_prob2_train = model2.predict_proba(x_train)[:, 1]
average_precision_score(y_train, y_prob2_train)